In [ ]:
#### This Code is used to pull in the data (dam locations and extracted temperature nodes), do intermediate processing, and organize the temperature profiles ####

In [ ]:
### Pull in Python Packages ###
import pandas as pd
import geopandas as gpd
import numpy as np
import glob
import os
from shapely.geometry import Point, Polygon
from scipy.spatial import cKDTree
import scipy.stats as stats
from tqdm.notebook import tqdm

In [ ]:
#######################################
##### Get the dam attribute information #####
#######################################

In [ ]:
## Pepare Dam Info##
# Pull in Study Dams
GrodDams = gpd.read_file(r"F:\Insert_File_Path_of_Shapefile\Study_Dams.shp", columns=['grod_id', 'type', 'lon', 'lat', 'hilarriid', 'dataset'])  # Update this file path

# Pull in NID Info
NID = pd.read_csv(r"F:\Insert_File_Path\NID_National_Data.csv", engine = 'python', skiprows=1, dtype={'Max Discharge (Cubic Ft/Second)': float}, 
                  usecols = ['Dam Name', 'NID ID','Primary Purpose', 'Latitude', 'Longitude', 'NID Height (Ft)', 'Normal Storage (Acre-Ft)', 'Surface Area (Acres)',
                             'Drainage Area (Sq Miles)', 'Max Discharge (Cubic Ft/Second)']) # Update this file path

# Remove points with no location info
NID = NID[~NID.Latitude.isna()]

# Make the NID spatial
NID_gpd = gpd.GeoDataFrame(NID, geometry=gpd.points_from_xy(NID.Longitude, NID.Latitude), crs="EPSG:4326")

In [ ]:
### Find the nearest dams ###
## Define Nearest Dam Function ##
def Closest_Dam(gdA, gdB):

    nA = np.array(list(gdA.geometry.apply(lambda x: (x.x, x.y))))
    nB = np.array(list(gdB.geometry.apply(lambda x: (x.x, x.y))))
    btree = cKDTree(nB)
    dist, idx = btree.query(nA, k=1)
    gdB_nearest = gdB.iloc[idx].drop(columns="geometry").reset_index(drop=True)
    gdf = pd.concat(
        [
            gdA.reset_index(drop=True),
            gdB_nearest,
            pd.Series(dist, name='dist')
        ], 
        axis=1)
    return gdf

## Find the Nearest Dam ##
DamSites = Closest_Dam(GrodDams,NID_gpd)
DamSites['NDist_m'] = (DamSites['dist']*111139) ## convert distance to meters from degrees

In [ ]:
# Select close and complete dams
Selected_Dams = DamSites[DamSites["NDist_m"]<=2000]
Selected_Dams = Selected_Dams.sort_values(by='grod_id', ascending=True)
Selected_Dams = Selected_Dams.reset_index(drop=True)
Selected_Dams_List = Selected_Dams["grod_id"].unique().tolist()
print("Number of Dams: " + str(len(Selected_Dams)))
Selected_Dams.head()

In [ ]:
# Export the df as a csv
Selected_Dams.to_csv(r"F:\Insert_File_Output_Path_Here\Selected_Dams_Site_Info.csv") # Update this file path

In [ ]:
#### Note: Need to clean up the data to only essenials
# Make a clean copy
Selected_Dams_Clean = Selected_Dams[['grod_id', 'lon', 'lat', 'Primary Purpose', 'NID Height (Ft)', 'Normal Storage (Acre-Ft)', 
                                     'Surface Area (Acres)', 'Drainage Area (Sq Miles)', 'Max Discharge (Cubic Ft/Second)','dataset']]
# Fix some column names
Selected_Dams_Clean = Selected_Dams_Clean.rename(columns={'grod_id': 'DamID', 'lon':'Longitude','lat':'Latitude'})

In [ ]:
## Export the Clean Data
Selected_Dams_Clean.to_csv(r"F:\Insert_File_Output_Path_Here\Clean_Dams_Site_Info.csv") # Update this file path

In [ ]:
###################################################
##### Get the profile/temperature change information #####
###################################################

In [ ]:
### Pull in all the temperatures collected ###
AllTempsCollected = pd.read_csv(r"F:\Insert_File_Output_Path_Here\Combined_Temps_Nodes.csv", 
                                engine='python', usecols= ['Join_Node', 'Month', 'Day', 'Year', 'Avg_Temp', 'Avg_RWC_Wid', 'x', 'y', 'reach_id',
                                                            'lakeflag', 'Assgn_dam', 'Dam_Flag', 'Up_Ds', 'Dam_Dist', 'Dam_Dist_km']) # Update this file path

In [ ]:
# Filter the temps to the dams with information, and points wider than 100m
Temps_Filtered = AllTempsCollected[AllTempsCollected["Assgn_dam"].isin(Selected_Dams_List)]
Temps_Filtered = Temps_Filtered[Temps_Filtered['Avg_RWC_Wid']>= 100]

# Flag the waterbody type up/ds
Temps_Filtered['Up_Ds_Grp']  = np.nan
Temps_Filtered['Up_Ds_Grp'] = np.where((Temps_Filtered['Up_Ds'] == 'Downstream') & (Temps_Filtered['lakeflag'] == 0) , 'Downstream River', Temps_Filtered['Up_Ds_Grp'])
Temps_Filtered['Up_Ds_Grp'] = np.where((Temps_Filtered['Up_Ds'] == 'Downstream') & (Temps_Filtered['lakeflag'] > 0) , 'Downstream Reservoir', Temps_Filtered['Up_Ds_Grp'])
Temps_Filtered['Up_Ds_Grp'] = np.where((Temps_Filtered['Up_Ds'] == 'Upstream') & (Temps_Filtered['lakeflag'] == 0) , 'Upstream River', Temps_Filtered['Up_Ds_Grp'])
Temps_Filtered['Up_Ds_Grp'] = np.where((Temps_Filtered['Up_Ds'] == 'Upstream') & (Temps_Filtered['lakeflag'] > 0) , 'Upstream Reservoir', Temps_Filtered['Up_Ds_Grp'])

# Preview Data
Temps_Filtered.head()

In [ ]:
### Get the usuable profiles -- First, 10km up/ds ###
## Identify where there are nodes upstream of the dam  
DirectUp_10= Temps_Filtered[(Temps_Filtered["Up_Ds"] == "Upstream") & (Temps_Filtered["Dam_Dist_km"] >= -10)]
DirectUp_10_grp = DirectUp_10.groupby(['Month','Day','Year','Assgn_dam']).agg({'Join_Node': ['count']})
DirectUp_10_grp.columns = ["NodeCount"]
DirectUp_10_grp = DirectUp_10_grp.reset_index()
## Identify Profiles with 5 Upstream River Nodes
DamTemps_DirUp10 = DirectUp_10_grp[DirectUp_10_grp["NodeCount"]>= 5]

## Identify where there are river nodes directly downstream from the dam ##
DsRiver_D10= Temps_Filtered[(Temps_Filtered["Up_Ds_Grp"] == "Downstream River") & (Temps_Filtered["Dam_Dist_km"] <= 10)]
DsRiver_D10_grp = DsRiver_D10.groupby(['Month','Day','Year','Assgn_dam']).agg({'Join_Node': ['count']})
DsRiver_D10_grp.columns = ["NodeCount"]
DsRiver_D10_grp = DsRiver_D10_grp.reset_index()
## Identify Profiles with 5 Downstream River Nodes
DamTemps_RivDs_10 = DsRiver_D10_grp[DsRiver_D10_grp["NodeCount"]>= 5]

#### NUMBER OF PROFILES ####
UpResAnDown_10 = pd.merge(DamTemps_RivDs_10, DamTemps_DirUp10, on = ['Assgn_dam', "Month", "Day", "Year"] , how='inner')
print("Number of Profiles: " + str(len(UpResAnDown_10)))
UpResAnDown_10

In [ ]:
# Export the results --- These profile dates will be used to pull the other variables
UpResAnDown_10.to_csv(r"F:\Insert_File_Output_Path_Here\Usable_Profiles_List.csv") # Update this file path

In [ ]:
## Get 'Significance' and Differences ## 
DamTemps_Up10Ds10 = Temps_Filtered[:]

## Calculate Significance (Up Dir Reserv/10km)##
col_names =  ['Assgn_dam','Month','Day','Year', 'Pval']
MWU_Prof_Results_Dir10  = pd.DataFrame(columns = col_names)
Unique_Profiles_Dir10 =  UpResAnDown_10[["Assgn_dam", "Month","Day", "Year"]]

for i in tqdm(range(len(Unique_Profiles_Dir10))):  # For selecting -- .iloc[i, 0] ## i for row , Dam Number (0), Month (1), Day (2), Year(3)
    # Set up the subsets for comparison -- All
    ManWU_1 = DamTemps_Up10Ds10[(DamTemps_Up10Ds10['Assgn_dam'] == Unique_Profiles_Dir10.iloc[i, 0]) & (DamTemps_Up10Ds10['Month'] == Unique_Profiles_Dir10.iloc[i, 1]) & (DamTemps_Up10Ds10['Day'] == Unique_Profiles_Dir10.iloc[i, 2]) & (DamTemps_Up10Ds10['Year'] == Unique_Profiles_Dir10.iloc[i, 3]) & (DamTemps_Up10Ds10['Up_Ds_Grp'] == "Downstream River") & (DamTemps_Up10Ds10['Dam_Dist_km'] <= 10)].Avg_Temp.tolist() # Downstream
    ManWU_2 = DamTemps_Up10Ds10[(DamTemps_Up10Ds10['Assgn_dam'] == Unique_Profiles_Dir10.iloc[i, 0]) & (DamTemps_Up10Ds10['Month'] == Unique_Profiles_Dir10.iloc[i, 1]) & (DamTemps_Up10Ds10['Day'] == Unique_Profiles_Dir10.iloc[i, 2]) & (DamTemps_Up10Ds10['Year'] == Unique_Profiles_Dir10.iloc[i, 3]) & (DamTemps_Up10Ds10['Up_Ds'] == "Upstream") & (DamTemps_Up10Ds10['Dam_Dist_km'] >= -10)].Avg_Temp.tolist() # Upstream    
     
    # Run the test
    statistic, p_value = stats.mannwhitneyu(ManWU_2,ManWU_1)
    
    # Get Dictionary
    dictionary = {'Assgn_dam': Unique_Profiles_Dir10.iloc[i, 0], 'Month': Unique_Profiles_Dir10.iloc[i, 1], 'Day': Unique_Profiles_Dir10.iloc[i, 2], 'Year': Unique_Profiles_Dir10.iloc[i, 3], 'Pval': [p_value]}
    
    df_dictionary = pd.DataFrame.from_dict(dictionary)
    
    # Add to DF
    output = pd.concat([MWU_Prof_Results_Dir10, df_dictionary], ignore_index=True)
    
    MWU_Prof_Results_Dir10 = output

# Define Significant profiles (Directly Up/10km)
MWU_Prof_Results_Dir10['Sig_05'] = np.where((MWU_Prof_Results_Dir10['Pval'] <= 0.05), 'Significant', 'Not Significant')
MWU_Prof_Results_Dir10

# Get the Significant Profiles within 10km
Signif_Dir10_Profiles = MWU_Prof_Results_Dir10[MWU_Prof_Results_Dir10["Sig_05"] == "Significant"]
print("Number of Profiles with Signficant differences Up/Ds: ", len(Signif_Dir10_Profiles))

### Percent Significant ### 
print("Percent Significant Profiles: ", len(Signif_Dir10_Profiles)/len(Unique_Profiles_Dir10))

## Add Significance to the nodes 
Signif_Info_Dir_10 = MWU_Prof_Results_Dir10[['Assgn_dam', 'Month', 'Day', 'Year', 'Sig_05']]

print("Nodes and significance:")
# Get ALL nodes, with signif info
All_Nodes_Dir10_Pval =  pd.merge(DamTemps_Up10Ds10, Signif_Info_Dir_10, on=['Month', 'Day','Year','Assgn_dam'], how='inner')
All_Nodes_Dir10_Pval

In [ ]:
## Get the Avg Diff Value for the profiles  ##
# Get the Profiles and Nodes -- Copy
Profiles_Dir10_Nodes = All_Nodes_Dir10_Pval[:]

# Of The Significant Profiles -- Avg Diff Up Vs Ds 
Upstream_Dir10 = Profiles_Dir10_Nodes[(Profiles_Dir10_Nodes['Up_Ds'] == "Upstream") & (Profiles_Dir10_Nodes["Dam_Dist_km"] >= -10)]
Upstream_Avgs_Dir10 = Upstream_Dir10.groupby(['Month','Day','Year','Assgn_dam']).agg({'Avg_Temp': ['mean']})
Upstream_Avgs_Dir10.columns = ['Up_Avg']
Upstream_Avgs_Dir10 = Upstream_Avgs_Dir10.reset_index()

Downstream_Riv_D10 = Profiles_Dir10_Nodes[(Profiles_Dir10_Nodes['Up_Ds_Grp'] == 'Downstream River') & (Profiles_Dir10_Nodes["Dam_Dist_km"] <= 10)]
Downstream_Avgs_Riv_D10 = Downstream_Riv_D10.groupby(['Month','Day','Year','Assgn_dam']).agg({'Avg_Temp': ['mean']})
Downstream_Avgs_Riv_D10.columns = ['Ds_Avg']
Downstream_Avgs_Riv_D10 = Downstream_Avgs_Riv_D10.reset_index()

Profile_Avgs_Dir10 = pd.merge(Upstream_Avgs_Dir10, Downstream_Avgs_Riv_D10, on=['Month','Day', 'Year','Assgn_dam'], how='outer' )

Profile_Avgs_Dir10['Temp_Diff'] = Profile_Avgs_Dir10['Ds_Avg'] - Profile_Avgs_Dir10['Up_Avg'] 
Profile_Avgs_Dir10['Abs_Temp_Diff'] = abs(Profile_Avgs_Dir10['Ds_Avg'] - Profile_Avgs_Dir10['Up_Avg'])

## Get Significance back on the DF 
Profile_Avgs_Dir_Sig_10 = pd.merge(Profile_Avgs_Dir10, Signif_Info_Dir_10, on=['Month', 'Day','Year','Assgn_dam'], how='inner')
Profile_Avgs_Dir_Sig_10

In [ ]:
# Export the results 
Profile_Avgs_Dir_Sig_10.to_csv(r"F:\Insert_File_Output_Path_Here\Avg_Profile_Change_Signif.csv") # Update this file path

In [ ]:
### Get the number of unique dams that have data and info from the NID ## 
# We'll also use to filter future outputs and subset this for reservoir only as well from this as well
Final_Dams_List = Profile_Avgs_Dir_Sig_10["Assgn_dam"].unique().tolist()
print("Number of Usable Dams: " + str(len(Final_Dams_List)))

In [ ]:
### From here, we need to get the data into a table that has Date(index), a YearMo column, then column for each dam's ds delta
# Get a proper date column
Daily_DS_Avg = Profile_Avgs_Dir_Sig_10[:]

# Fix Padded Zeros Issue
Daily_DS_Avg['Month_Pad'] = Daily_DS_Avg['Month'].astype(str)
Daily_DS_Avg['Month_Pad'] = Daily_DS_Avg['Month_Pad'].str.pad(width=2, side='left', fillchar='0')
Daily_DS_Avg['Day_Pad'] = Daily_DS_Avg['Day'].astype(str)
Daily_DS_Avg['Day_Pad'] = Daily_DS_Avg['Day_Pad'].str.pad(width=2, side='left', fillchar='0')

# Create columns for index
Daily_DS_Avg['Date'] = Daily_DS_Avg[['Year', 'Month_Pad', 'Day_Pad']].apply(lambda row: '-'.join(row.values.astype(str)), axis=1)
Daily_DS_Avg['Date'] = pd.to_datetime(Daily_DS_Avg['Date'], format ='%Y-%m-%d')
Daily_DS_Avg['YearMo']= Daily_DS_Avg[['Year', 'Month_Pad']].apply(lambda row: ''.join(row.values.astype(str)), axis=1)

# Clean up and Fix Index
Daily_DS_Avg = Daily_DS_Avg[['Date','YearMo','Month','Assgn_dam', 'Up_Avg', 'Ds_Avg', 'Temp_Diff', 'Abs_Temp_Diff', 'Sig_05' ]]
Daily_DS_Avg

In [ ]:
# Export the results 
Daily_DS_Avg.to_csv(r"F:\Insert_File_Output_Path_Here\Avg_Profile_Changes_Final.csv") # Update this file path

In [ ]:
################################
###### Get Final Dam Site Info  ######
################################

In [ ]:
# Filter the site information to just the dams we want 
FinalDamInfo = Selected_Dams_Clean[Selected_Dams_Clean["DamID"].isin(Final_Dams_List)]

# Export the Final Dam Info
FinalDamInfo.to_csv(r"F:\Insert_File_Output_Path_Here\Study_Dams_Site_Info.csv") # Update this file path

In [ ]:
######## After this script is run: Information is pulled from Daymet and RTMA (using the profiles identified in this script). 
## This and the Caravan information must be completed before moving on to "2_Prepare_Tables_for_ML" ##